In [2]:
from pathlib import Path
import os, sys, subprocess, textwrap, shutil, re

WORK = Path("/kaggle/working")
YOLOV7_DIR = WORK / "yolov7"
WEIGHTS = WORK / "yolov7.pt"

if not YOLOV7_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/WongKinYiu/yolov7.git", str(YOLOV7_DIR)],
        check=True
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "thop", "pycocotools", "pyyaml"],
    check=True
)

os.chdir(YOLOV7_DIR)

# Patch nhỏ cho PyTorch mới khi load checkpoint YOLOv7 cũ
for rel in ["models/experimental.py", "train.py"]:
    p = YOLOV7_DIR / rel
    s = p.read_text()
    s = s.replace(
        "torch.load(w, map_location=map_location)",
        "torch.load(w, map_location=map_location, weights_only=False)"
    )
    s = s.replace(
        "torch.load(weights, map_location=device)",
        "torch.load(weights, map_location=device, weights_only=False)"
    )
    p.write_text(s)

if not WEIGHTS.exists():
    subprocess.run(
        ["wget", "-q", "https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt", "-O", str(WEIGHTS)],
        check=True
    )

print("YOLOv7 ready:", YOLOV7_DIR)
print("Weights:", WEIGHTS)

Cloning into '/kaggle/working/yolov7'...


YOLOv7 ready: /kaggle/working/yolov7
Weights: /kaggle/working/yolov7.pt


In [3]:
from pathlib import Path
import os, shutil, yaml, pandas as pd

INPUT = Path("/kaggle/input")
WORK_DATA = Path("/kaggle/working/garbage_dataset_yolov7")
DATA_YAML = Path("/kaggle/working/garbage_yolov7.yaml")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
NAMES_DEFAULT = ["BIODEGRADABLE", "CARDBOARD", "GLASS", "METAL", "PAPER", "PLASTIC"]

def count_imgs(p):
    return sum(1 for x in p.glob("*") if x.suffix.lower() in IMG_EXTS) if p.exists() else 0

def count_labels(p):
    return sum(1 for x in p.glob("*.txt")) if p.exists() else 0

candidates = []
for train_images in INPUT.rglob("train/images"):
    root = train_images.parent.parent
    if not (root / "valid/images").exists():
        continue

    candidates.append({
        "root": root,
        "name": root.name,
        "train_images": count_imgs(root / "train/images"),
        "valid_images": count_imgs(root / "valid/images"),
        "test_images": count_imgs(root / "test/images"),
        "train_labels": count_labels(root / "train/labels"),
        "valid_labels": count_labels(root / "valid/labels"),
        "test_labels": count_labels(root / "test/labels"),
    })

df = pd.DataFrame(candidates)
display(df)

good = df[
    (df["train_labels"] > 0) &
    (df["valid_labels"] > 0) &
    (df["test_labels"] > 0)
].copy()

assert len(good), "Không tìm thấy folder có đủ labels. Hãy chọn bản yolov5pytorch/yolov8/yolov9."

priority = ["yolov7pytorch", "yolov5pytorch", "yolov8", "yolov9"]
good["priority"] = good["name"].apply(
    lambda n: min([i for i, k in enumerate(priority) if k in n] or [99])
)

SRC_DATASET_DIR = good.sort_values(["priority", "train_labels"], ascending=[True, False]).iloc[0]["root"]
print("Selected source:", SRC_DATASET_DIR)

# Tạo wrapper writable trong /kaggle/working.
if WORK_DATA.exists():
    shutil.rmtree(WORK_DATA)

for split in ["train", "valid", "test"]:
    (WORK_DATA / split).mkdir(parents=True, exist_ok=True)
    os.symlink(SRC_DATASET_DIR / split / "images", WORK_DATA / split / "images")
    os.symlink(SRC_DATASET_DIR / split / "labels", WORK_DATA / split / "labels")

src_yaml = SRC_DATASET_DIR / "data.yaml"
names = NAMES_DEFAULT
if src_yaml.exists():
    raw = yaml.safe_load(src_yaml.read_text())
    names = raw.get("names", names)

fixed_yaml = {
    "train": str(WORK_DATA / "train/images"),
    "val": str(WORK_DATA / "valid/images"),
    "test": str(WORK_DATA / "test/images"),
    "nc": len(names),
    "names": names,
}

DATA_YAML.write_text(yaml.safe_dump(fixed_yaml, sort_keys=False))

print(DATA_YAML.read_text())
assert "/kaggle/input" not in fixed_yaml["train"], "YAML vẫn đang trỏ vào input read-only"

,root,name,train_images,valid_images,test_images,train_labels,valid_labels,test_labels
0,/kaggle/input/datasets/minhkhoathi/garbage-cla...,GARBAGE CLASSIFICATION 3.v2-gc1.yolo26,7324,2098,1042,7324,2098,1042
1,/kaggle/input/datasets/minhkhoathi/garbage-cla...,GARBAGE CLASSIFICATION 3.v2-gc1.yolov11,7324,2098,1042,7324,2098,1042
2,/kaggle/input/datasets/minhkhoathi/garbage-cla...,GARBAGE CLASSIFICATION 3.v2-gc1.yolov8,7324,2098,1042,7324,2098,1042
3,/kaggle/input/datasets/minhkhoathi/garbage-cla...,GARBAGE CLASSIFICATION 3.v2-gc1.yolov5pytorch,7324,2098,1042,7324,2098,1042
4,/kaggle/input/datasets/minhkhoathi/garbage-cla...,GARBAGE CLASSIFICATION 3.v2-gc1.yolov12,7324,2098,1042,7324,2098,1042
5,/kaggle/input/datasets/minhkhoathi/garbage-cla...,GARBAGE CLASSIFICATION 3.v2-gc1.yolov9,7324,2098,1042,7324,2098,1042
6,/kaggle/input/datasets/minhkhoathi/garbage-cla...,GARBAGE CLASSIFICATION 3.v2-gc1.yolov7pytorch,7324,2098,1042,0,0,0


Selected source: /kaggle/input/datasets/minhkhoathi/garbage-classification/GARBAGE CLASSIFICATION 3.v2-gc1.yolov5pytorch
train: /kaggle/working/garbage_dataset_yolov7/train/images
val: /kaggle/working/garbage_dataset_yolov7/valid/images
test: /kaggle/working/garbage_dataset_yolov7/test/images
nc: 6
names:
- BIODEGRADABLE
- CARDBOARD
- GLASS
- METAL
- PAPER
- PLASTIC



In [9]:
import os, sys, subprocess, torch
from pathlib import Path

os.chdir(YOLOV7_DIR)

os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_SILENT"] = "true"

IMG_SIZE = 416
BATCH = 16
EPOCHS = 50

RUN_DIR = Path("/kaggle/input/models/minhkhoathi/yolo-7-garbage-object-detection/pytorch/default/1")
LAST_WEIGHTS = RUN_DIR / "last.pt"
BEST_WEIGHTS = RUN_DIR / "best.pt"

device = "0" if torch.cuda.is_available() else "cpu"

if LAST_WEIGHTS.exists():
    print("Resuming from:", LAST_WEIGHTS)
    cmd = [
        sys.executable, "train.py",
        "--resume", str(LAST_WEIGHTS),
    ]
else:
    print("Starting new training")
    cmd = [
        sys.executable, "train.py",
        "--workers", "2",
        "--device", device,
        "--batch-size", str(BATCH),
        "--epochs", str(EPOCHS),
        "--img-size", str(IMG_SIZE), str(IMG_SIZE),
        "--data", str(DATA_YAML),
        "--weights", str(WEIGHTS),
        "--project", "/kaggle/working/runs/train",
        "--name", RUN_NAME,
        "--exist-ok",
    ]

env = os.environ.copy()
env["WANDB_MODE"] = "disabled"
env["WANDB_DISABLED"] = "true"
env["WANDB_SILENT"] = "true"

result = subprocess.run(
    cmd,
    check=False,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(result.stdout)

if result.returncode != 0:
    print("\n===== LAST 120 LOG LINES =====")
    print("\n".join(result.stdout.splitlines()[-120:]))
    raise RuntimeError("train.py failed. Xem LAST 120 LOG LINES ở trên để biết lỗi thật.")

print("Best weights:", BEST_WEIGHTS)

Starting new training


NameError: name 'RUN_NAME' is not defined

In [ ]:
EVAL_NAME = "benchmark_yolov7_test"
PRED_DIR = Path("/kaggle/working/runs/test") / EVAL_NAME / "labels"

cmd = [
    sys.executable, "test.py",
    "--data", str(DATA_YAML),
    "--weights", str(BEST_WEIGHTS),
    "--img-size", str(IMG_SIZE),
    "--batch-size", str(BATCH),
    "--task", "test",
    "--conf-thres", "0.001",
    "--iou-thres", "0.65",
    "--save-txt",
    "--save-conf",
    "--verbose",
    "--project", "/kaggle/working/runs/test",
    "--name", EVAL_NAME,
    "--exist-ok",
]

subprocess.run(cmd, check=True)
print("Pred labels:", PRED_DIR)

In [7]:
from pathlib import Path
import os, sys, subprocess, torch

YOLOV7_DIR = Path("/kaggle/working/yolov7")
DATA_YAML = Path("/kaggle/working/garbage_yolov7.yaml")

BEST_WEIGHTS =  Path('/kaggle/input/models/minhkhoathi/yolo-7-garbage-object-detection/pytorch/default/2/best.pt')

print("DATA_YAML exists:", DATA_YAML.exists(), DATA_YAML)
print("BEST_WEIGHTS exists:", BEST_WEIGHTS.exists(), BEST_WEIGHTS)

assert DATA_YAML.exists(), "Chưa có garbage_yolov7.yaml. Hãy chạy Cell 2 trước."
assert BEST_WEIGHTS.exists(), "Chưa thấy best.pt. Kiểm tra lại training folder hoặc dùng last.pt."

print(DATA_YAML.read_text())

DATA_YAML exists: True /kaggle/working/garbage_yolov7.yaml
BEST_WEIGHTS exists: True /kaggle/input/models/minhkhoathi/yolo-7-garbage-object-detection/pytorch/default/2/best.pt
train: /kaggle/working/garbage_dataset_yolov7/train/images
val: /kaggle/working/garbage_dataset_yolov7/valid/images
test: /kaggle/working/garbage_dataset_yolov7/test/images
nc: 6
names:
- BIODEGRADABLE
- CARDBOARD
- GLASS
- METAL
- PAPER
- PLASTIC



In [10]:
from pathlib import Path
import os, sys, subprocess

YOLOV7_DIR = Path("/kaggle/working/yolov7")
DATA_YAML = Path("/kaggle/working/garbage_yolov7.yaml")
BEST_WEIGHTS = Path("/kaggle/input/models/minhkhoathi/yolo-7-garbage-object-detection/pytorch/default/2/best.pt")

assert YOLOV7_DIR.exists(), "Chưa có YOLOv7 repo. Hãy chạy cell setup YOLOv7 trước."
assert DATA_YAML.exists(), "Chưa có garbage_yolov7.yaml. Hãy chạy Cell 2 tạo data yaml trước."
assert BEST_WEIGHTS.exists(), "Không tìm thấy best.pt, kiểm tra lại path model."

os.chdir(YOLOV7_DIR)

IMG_SIZE = 416
BATCH = 16
EVAL_NAME = "benchmark_yolov7_best_test"
PRED_DIR = Path("/kaggle/working/runs/test") / EVAL_NAME / "labels"

cmd = [
    sys.executable, "test.py",
    "--data", str(DATA_YAML),
    "--weights", str(BEST_WEIGHTS),
    "--img-size", str(IMG_SIZE),
    "--batch-size", str(BATCH),
    "--task", "test",
    "--conf-thres", "0.001",
    "--iou-thres", "0.65",
    "--save-txt",
    "--save-conf",
    "--verbose",
    "--project", "/kaggle/working/runs/test",
    "--name", EVAL_NAME,
    "--exist-ok",
]

result = subprocess.run(
    cmd,
    check=False,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(result.stdout)

if result.returncode != 0:
    print("\n===== LAST 120 LOG LINES =====")
    print("\n".join(result.stdout.splitlines()[-120:]))
    raise RuntimeError("test.py failed. Xem lỗi thật ở trên.")

print("PRED_DIR:", PRED_DIR)
print("Pred files:", len(list(PRED_DIR.glob("*.txt"))))

YOLOR 🚀 v0.1-128-ga207844 torch 2.10.0+cu128 CUDA:0 (Tesla T4, 14911.6875MB)
                                             CUDA:1 (Tesla T4, 14911.6875MB)

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Model Summary: 306 layers, 36506851 parameters, 6194944 gradients, 103.2 GFLOPS
Namespace(weights=['/kaggle/input/models/minhkhoathi/yolo-7-garbage-object-detection/pytorch/default/2/best.pt'], data='/kaggle/working/garbage_yolov7.yaml', batch_size=16, img_size=416, conf_thres=0.001, iou_thres=0.65, task='test', device='', single_cls=False, augment=False, verbose=True, save_txt=True, save_hybrid=False, save_conf=True, save_json=False, project='/kaggle/working/runs/test', name='benchmark_yolov7_best_test', exist_ok=True, n

In [11]:
from pathlib import Path
from PIL import Image
from collections import defaultdict
import numpy as np, pandas as pd, yaml

CONF_FOR_PRF = 0.25
IOU_FOR_PRF = 0.50
IOU_THRESHOLDS = np.arange(0.50, 0.96, 0.05)

data = yaml.safe_load(Path(DATA_YAML).read_text())
names = data["names"]
img_dir = Path(data["test"])
label_dir = img_dir.parent / "labels"

def yolo_to_xyxy(x, y, w, h, iw, ih):
    return np.array([
        (x - w / 2) * iw,
        (y - h / 2) * ih,
        (x + w / 2) * iw,
        (y + h / 2) * ih,
    ], dtype=float)

def box_iou(box, boxes):
    if len(boxes) == 0:
        return np.array([])
    boxes = np.asarray(boxes)
    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    area1 = max(0, box[2] - box[0]) * max(0, box[3] - box[1])
    area2 = np.maximum(0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0, boxes[:, 3] - boxes[:, 1])
    return inter / (area1 + area2 - inter + 1e-9)

def read_yolo(path, size, pred=False):
    out = []
    if not path.exists():
        return out
    iw, ih = size
    for line in path.read_text().splitlines():
        if not line.strip():
            continue
        vals = list(map(float, line.split()))
        cls = int(vals[0])
        box = yolo_to_xyxy(*vals[1:5], iw, ih)
        conf = vals[5] if pred and len(vals) > 5 else 1.0
        out.append((cls, box, conf))
    return out

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([1.0], precision, [0.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101)
    return np.trapz(np.interp(x, mrec, mpre), x)

image_paths = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS])

gt_by_cls = {i: defaultdict(list) for i in range(len(names))}
pred_by_cls = {i: [] for i in range(len(names))}
n_gt = np.zeros(len(names), dtype=int)

for img_path in image_paths:
    stem = img_path.stem
    with Image.open(img_path) as im:
        size = im.size

    for cls, box, _ in read_yolo(label_dir / f"{stem}.txt", size, pred=False):
        if 0 <= cls < len(names):
            gt_by_cls[cls][stem].append(box)
            n_gt[cls] += 1

    for cls, box, conf in read_yolo(PRED_DIR / f"{stem}.txt", size, pred=True):
        if 0 <= cls < len(names):
            pred_by_cls[cls].append((stem, conf, box))

def match_class(cls, iou_thr, min_conf=0.0):
    preds = [p for p in pred_by_cls[cls] if p[1] >= min_conf]
    preds = sorted(preds, key=lambda x: x[1], reverse=True)

    matched = {
        img_id: np.zeros(len(boxes), dtype=bool)
        for img_id, boxes in gt_by_cls[cls].items()
    }

    tp, fp, matched_ious = [], [], []

    for img_id, conf, box in preds:
        gt_boxes = gt_by_cls[cls].get(img_id, [])
        ious = box_iou(box, gt_boxes)

        if len(ious) and ious.max() >= iou_thr:
            j = int(ious.argmax())
            if not matched[img_id][j]:
                matched[img_id][j] = True
                tp.append(1)
                fp.append(0)
                matched_ious.append(float(ious[j]))
            else:
                tp.append(0)
                fp.append(1)
        else:
            tp.append(0)
            fp.append(1)

    return np.array(tp), np.array(fp), matched_ious

rows = []
ap_matrix = np.zeros((len(names), len(IOU_THRESHOLDS)))

for c, name in enumerate(names):
    for j, thr in enumerate(IOU_THRESHOLDS):
        tp, fp, _ = match_class(c, thr, min_conf=0.0)
        if n_gt[c] == 0:
            ap = np.nan
        elif len(tp) == 0:
            ap = 0.0
        else:
            tp_cum = np.cumsum(tp)
            fp_cum = np.cumsum(fp)
            recall = tp_cum / (n_gt[c] + 1e-9)
            precision = tp_cum / (tp_cum + fp_cum + 1e-9)
            ap = compute_ap(recall, precision)
        ap_matrix[c, j] = ap

    tp, fp, matched_ious = match_class(c, IOU_FOR_PRF, min_conf=CONF_FOR_PRF)
    TP = int(tp.sum())
    FP = int(fp.sum())
    FN = int(n_gt[c] - TP)

    precision = TP / (TP + FP + 1e-9)
    recall = TP / (TP + FN + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    rows.append({
        "class": name,
        "gt": int(n_gt[c]),
        "tp": TP,
        "fp": FP,
        "fn": FN,
        f"precision@conf{CONF_FOR_PRF}": precision,
        f"recall@conf{CONF_FOR_PRF}": recall,
        f"f1@conf{CONF_FOR_PRF}": f1,
        "mean_iou_true_positive": np.mean(matched_ious) if matched_ious else np.nan,
        "AP50": ap_matrix[c, 0],
        "AP50_95": np.nanmean(ap_matrix[c]),
    })

metrics_df = pd.DataFrame(rows)

TP = metrics_df["tp"].sum()
FP = metrics_df["fp"].sum()
FN = metrics_df["fn"].sum()

summary = pd.DataFrame([{
    "model": "YOLOv7",
    "eval_split": "test",
    "iou_for_precision_recall_f1": IOU_FOR_PRF,
    "conf_for_precision_recall_f1": CONF_FOR_PRF,
    "micro_precision": TP / (TP + FP + 1e-9),
    "micro_recall": TP / (TP + FN + 1e-9),
    "micro_f1": 2 * TP / (2 * TP + FP + FN + 1e-9),
    "mean_iou_true_positive": metrics_df["mean_iou_true_positive"].mean(),
    "mAP50": np.nanmean(ap_matrix[:, 0]),
    "mAP50_95": np.nanmean(ap_matrix),
}])

display(summary)
display(metrics_df)

metrics_df.to_csv("/kaggle/working/yolov7_per_class_metrics.csv", index=False)
summary.to_csv("/kaggle/working/yolov7_summary_metrics.csv", index=False)

print("Saved:")
print("/kaggle/working/yolov7_per_class_metrics.csv")
print("/kaggle/working/yolov7_summary_metrics.csv")

/tmp/ipykernel_58/2892130392.py:56: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(np.interp(x, mrec, mpre), x)
/tmp/ipykernel_58/2892130392.py:147: RuntimeWarning: Mean of empty slice
  "AP50_95": np.nanmean(ap_matrix[c]),


,model,eval_split,iou_for_precision_recall_f1,conf_for_precision_recall_f1,micro_precision,micro_recall,micro_f1,mean_iou_true_positive,mAP50,mAP50_95
0,YOLOv7,test,0.5,0.25,0.58523,0.649453,0.615671,0.830451,0.467565,0.348642


,class,gt,tp,fp,fn,precision@conf0.25,recall@conf0.25,f1@conf0.25,mean_iou_true_positive,AP50,AP50_95
0,BIODEGRADABLE,49,19,194,30,0.089202,0.387755,0.145038,0.726273,0.105822,0.058675
1,CARDBOARD,20,7,70,13,0.090909,0.350000,0.144330,0.804607,0.138373,0.107495
2,GLASS,0,0,86,0,0.000000,0.000000,0.000000,NaN,NaN,NaN
3,METAL,533,384,167,149,0.696915,0.720450,0.708487,0.904687,0.709510,0.575800
4,PAPER,1376,869,437,507,0.665391,0.631541,0.648024,0.873530,0.695273,0.523852
5,PLASTIC,1585,1035,686,550,0.601395,0.652997,0.626134,0.843159,0.688845,0.477388


Saved:
/kaggle/working/yolov7_per_class_metrics.csv
/kaggle/working/yolov7_summary_metrics.csv
